# Inverse Problems Exercises: 2023s s04 (non-sc)
https://www.umm.uni-heidelberg.de/miism/

## Notes
* Please **DO NOT** change the name of the `.ipynb` file. 
* Please **DO NOT** import extra packages to solve the tasks. 
* Please put the `.ipynb` file directly into the `.zip` archive without any intermediate folder. 

## Please provide your personal information
* full name (Name): 

Jan Benda

## D07: Singular value decomposition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from numpy import linalg as LA

In [ ]:
file_gaussian = 'file_gaussian.npz'
with np.load(file_gaussian) as data:
    f_true = data['f_true']
    A_psf = data['A_psf']
    list_gn = data['list_gn']

### Imaging model
The imaging model can be represented by
$$
g = h \otimes f_\text{true}
= Af_\text{true}
= \mathcal{F}^{-1}\{ \mathcal{F}\{h\} \mathcal{F}\{f_\text{true}\} \},
$$
$$
g' = g + \epsilon.
$$
* $f_\text{true}$ is the input signal
* $h$ is the point spread function (kernel)
* $\otimes$ is the convolution operator
* $A$ is the Toeplitz matrix of $h$
* $\mathcal{F}$ and $\mathcal{F}^{-1}$ are the Fourier transform operator and inverse Fourier transform operator
* $\epsilon$ is the additive Gaussian noise
* $g$ is the filtered signal
* $g'$ is the noisy signal

### Frobenius norm
Implement the Frobenius norm by definition
$$
\|A\|_\text{F} = \sqrt{\sum_{i}^m\sum_{j}^n |a_{ij}|^2},
$$
where $A$ is an $m \times n$ matrix. When $m = 1$ or $n = 1$, $A$ is a vector.
* Given the matrix $A$
* Calculate the Frobenius norm (`numpy.linalg.norm()` should NOT be used.)
* Implement the function `frobenius_norm()`

In [ ]:
def frobenius_norm(mat_A):
    """ Compute the Frobenius norm of the matrix:

    :param mat_A: Input matrix or vector.
    :returns: Frobenius norm.
    """
    return np.sqrt(np.sum(mat_A*mat_A))

In [ ]:
# This cell contains hidden tests.


### Condition number of $A$

Calculate the condition number of matrix $A$
$$
\operatorname{cond}_\text{F}(A) = \|A^{-1}\|_\text{F} \|A\|_\text{F}
$$
* Apply the calculation to `A_psf`
* Save $\|A\|_\text{F}$ in the variable `norm_A_psf` (using `frobenius_norm()`)
* Save $\|A^{-1}\|_\text{F}$ in the variable `norm_A_inv` (using `frobenius_norm()`)
* Save $\operatorname{cond}_\text{F}(A)$ in the variable `cond_A_psf` (`numpy.linalg.cond()` should not be used.)

In [ ]:
norm_A_psf = frobenius_norm(A_psf)
norm_A_inv = frobenius_norm(LA.inv(A_psf))
cond_A_psf = norm_A_inv*norm_A_psf


In [ ]:
# This cell contains tests.

print(cond_A_psf)


### Question: Condition number of $A$
Is the inversion of the system with `A_psf` stable?

It is most likely not stable, since the condition number of `A_psf` is huge. Therefore any noise in the right hand side will be amplified drastically.

### Singular value decomposition

Calculate the singular value decomposition (SVD) of real matrix $A$
$$
A = USV^T,
$$
where $UU^T = VV^T = I$, and $S$ is a rectangular diagonal matrix with non-negative real numbers on the diagonal. The entries on the diagonal $\operatorname{diag}(S)_i$ are the singular values.

* Apply the calculation to `A_psf` (using `numpy.linalg.svd()`)
* Plot $A$, $U$, $S$ and $V^T$ in order in the subplots of `axs`
* Plot the matrices $A$, $U$, and $V^T$ as grayscale images in the corresponding subplots
* Show the colorbar of the above three subplots
* Plot the singular values $\operatorname{diag}(S)_i$ as a line in the third subplot
* Add proper titles to the subplots of `axs`

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 5))
fig.suptitle('Singular value decomposition')

U,S,V_T = np.linalg.svd(A_psf)
cax = 4*[None]
cax[0] = axs[0].imshow(A_psf,cmap='gray')
cax[1] = axs[1].imshow(U,cmap='gray')
cax[2] = axs[2].imshow(np.diag(S),cmap='gray')
cax[3] = axs[3].imshow(V_T,cmap='gray')

axs[0].set_title(r"Orignal matrix $A$")
axs[1].set_title(r"First unitary matrix $U$")
axs[2].set_title(r"Singular values (the diagonal of $S$)")
axs[3].set_title(r"Second unitary matrix $V^T$")


for i in range(4):
    plt.colorbar(cax[i], ax=axs[i], location = "top", shrink = 0.6,pad=0.1)


In [ ]:
# This cell contains hidden tests.


### Condition number of $S$

Calculate the condition number of matrix $S$
* Apply the calculation to the $S$ from SVD 
* Save $\|S\|_\text{F}$ in the variable `norm_S_psf` (using `frobenius_norm()`)
* Save $\|S^{-1}\|_\text{F}$ in the variable `norm_S_inv` (using `frobenius_norm()`)
* Save $\operatorname{cond}_\text{F}(S)$ in the variable `cond_S_psf` (`numpy.linalg.cond()` should not be used.)

In [ ]:
# YOUR CODE HERE
norm_S_psf = frobenius_norm(S)
norm_S_inv = frobenius_norm(1/S)
cond_S_psf = norm_S_psf*norm_S_inv

In [ ]:
# This cell contains tests.

print(cond_S_psf)


### Question: Condition number of $S$
Is the equation $\operatorname{cond}_\text{F}(A) = \operatorname{cond}_\text{F}(S)$ valid? Why?

It is valid, because in both cases we have calculated the same norms. And that is, because multiplying some matrix $A$ by a unitary matrix $Q$ does not change the norm of $A$. Here is the proof.
$$||QA||_\text{F}^2 = \sum_i ||QA_i||= \sum_i ||A_i|| = ||A||_\text{F}^2\,,$$
where $A_i$ is the $i$-th column of $A$ and we sum over all of its columns. Therefore
$$||A||_\text{F}=||USV^\text{T}||_\text{F}=||S||_\text{F}$$

### Truncated singular values

$S_t$ contains the truncated singular values as
$$
\operatorname{diag}(S_t)_i = 
\begin{cases}
    \operatorname{diag}(S)_i & \operatorname{diag}(S)_i \geq \text{TH}\\
    1 & \text{otherwise} \\
\end{cases},
$$
i.e. to set the singular values in $S$ less than the threshold $\text{TH}$ to $1$.

* Set $\text{TH}$ with $2\% \cdot \max(\operatorname{diag}(S)_i)$, $10\% \cdot \max(\operatorname{diag}(S)_i)$, $50\% \cdot \max(\operatorname{diag}(S)_i)$, respectively
* Generate $S_t$
* Save $A_t = U S_t V^T$ in the variable `list_A_tsvd` (as `list` of `numpy.array`)
* Save $\operatorname{cond}_\text{F}(A_t)$ in the variable `list_cond_A_tsvd` (`numpy.linalg.cond()` should not be used.)

In [ ]:
def trunc(S,p):
    Snew = S.copy()
    for i in range(len(S)):
        if Snew[i]<Snew[0]*p:
            Snew[i] = 1
    return Snew


list_A_tsvd = [U @ np.diag(trunc(S,p)) @V_T for p in [0.02, 0.1, 0.5]]
list_cond_A_tsvd = [frobenius_norm(A)*frobenius_norm(LA.inv(A)) for A in list_A_tsvd]




In [ ]:
# This cell contains tests.

print(list_cond_A_tsvd)


### Reconstruction

Reconstruct the signal by
$$
\tilde{f} = A_t^{-1} g'
$$
* Apply this operation to the noisy signals in `list_gn`
* Return the outputs with different $A_t$ in `list_A_tsvd`
* Save the outputs in the variable `list_f_tsvd` (as `list` of `numpy.array`)

Display the result
* Plot the outputs in `list_f_tsvd` in the same order of the noisy signals in the subplots of `axs`
* Show the cases of the same noisy signal in the same subplot column
* Show the cases with the same $A_t$ in the same subplot row
* Plot the corresponding noisy signal in each subplot (after the filter output)
* Plot the input signal `f_true` in each subplot (after the noisy signal)
* Show the legend in each subplot
* Show the case information in the titles to the subplots

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Reconstruction')
list_f_tsvd = []
for x in range(3):
    for y in range(3):
        f_tilde = LA.inv(list_A_tsvd[x])@list_gn[y]
        list_f_tsvd.append(f_tilde)
        
        ax = axs[x][y]
        ax.plot(f_tilde, label = "filtered data")
        ax.plot(f_true, label = "input signal")
        ax.set_title("signal "+str(y+1)+r", $A_t$ calculated with $p = {p:.2f}$".format(p=[0.02, 0.1, 0.5][x]))
        ax.legend()
        


In [ ]:
# This cell contains hidden tests.


### Question: Truncated singular values

Describe the visual effect on the reconstruction result considering the influence of $\text{TH}$.

Increasind the treshold decreased the condition number of $A_t$ which in turn hat positive effect on noise reduction.
On the other hand, hight treshold causes a loss of information about the original data, since our method cannot recognize the difference between the noise and the detail in data. So for less noisy data the method with lower treshold works better.